In [2]:
import pandas as pd
import networkx as nx
import numpy as np
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

In [3]:
##get Simulated E4.5 epiblast data
expression_df = pd.read_csv('Generated_epi_data.csv')

In [23]:
expression_df

,Unnamed: 0,Phlda2,Gjb2,Rhox6,Rhox9,Slc13a4,Maged1,Krt8,Col4a1,Itm2a,...,Fcgrt,Scpep1,Foxj1,Rassf1,Deptor,Atp5g2,Ptdss1,Dock4,Cpt1a,Tmem47
0,0,0.113611,0.0,0.0,0,0.366991,0.000000,0.000000,0.000000,0.0,...,0.000000,0.759939,0,0.000000,0.140574,2.238391,0.004359,0.000000,0.531835,0.000000
1,1,0.000000,0.0,0.0,0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.000000,0,0.000000,0.000000,2.033418,0.245176,1.651925,0.000000,0.000000
2,2,0.000000,0.0,0.0,0,0.548954,0.000000,0.000000,0.000000,0.0,...,0.090232,0.141355,0,0.000000,0.888523,1.952406,0.000000,0.000000,0.161413,0.000000
3,3,0.164112,0.0,0.0,0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.513730,0,0.000000,0.448428,1.635942,0.000000,0.000000,0.000000,0.000000
4,4,0.000000,0.0,0.0,0,0.000000,0.157351,0.000000,0.000000,0.0,...,0.000000,0.253454,0,0.000000,0.386118,1.994791,0.000000,0.492300,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,495,0.000000,0.0,0.0,0,0.000000,0.333257,0.899614,0.000000,0.0,...,0.000000,0.435652,0,0.458551,0.000000,1.159298,0.000000,0.559459,0.000000,0.148197
496,496,0.000000,0.0,0.0,0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.982120,0,0.000000,0.147385,1.563528,0.194975,0.237197,0.000000,0.000000
497,497,0.000000,0.0,0.0,0,0.000000,0.429372,1.008962,1.478615,0.0,...,0.000000,0.000000,0,0.000000,0.482641,1.279138,0.000000,0.551390,0.000000,0.000000
498,498,0.288202,0.0,0.0,0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.010379,0,0.000000,0.443766,2.122722,0.000000,0.266555,0.000000,0.000000


In [3]:
gene_names = list(expression_df.columns)

In [4]:
#calculated gene-coexpression 
from sklearn.covariance import EmpiricalCovariance
# use the expression of 2000 hvgs between E3.5 to E8.5
large_expression_df = pd.read_csv('combined_expression.csv', index_col=0)
cov_model = EmpiricalCovariance().fit(large_expression_df)
cov_matrix = pd.DataFrame(cov_model.covariance_, index=large_expression_df.columns, columns=large_expression_df.columns)


In [5]:
cov_matrix

,Phlda2,Gjb2,Rhox6,Rhox9,Slc13a4,Maged1,Krt8,Col4a1,Itm2a,Serpine1,...,Fcgrt,Scpep1,Foxj1,Rassf1,Deptor,Atp5g2,Ptdss1,Dock4,Cpt1a,Tmem47
Phlda2,1.569503,0.008971,0.104692,0.071038,0.027365,0.019883,0.452234,-0.079069,-0.024965,0.010428,...,-0.003081,-0.013374,-0.014119,0.010513,-0.031205,0.029159,-0.005043,-0.028147,-0.008071,-0.015689
Gjb2,0.008971,0.019646,0.004253,-0.000601,0.014746,-0.001550,0.013669,0.011952,-0.004277,-0.000296,...,0.005258,0.011550,0.000036,0.001367,-0.000738,-0.002025,0.002249,-0.000134,0.000137,-0.002508
Rhox6,0.104692,0.004253,0.146466,0.061790,0.013841,-0.032076,0.080563,0.031513,-0.026495,0.003006,...,0.007172,0.022675,-0.001595,0.007191,-0.000657,-0.019998,0.010930,-0.001086,0.001604,-0.017346
Rhox9,0.071038,-0.000601,0.061790,0.068658,-0.003024,-0.023647,0.041686,-0.003745,-0.011997,0.001800,...,-0.000440,-0.000787,-0.000883,0.002073,-0.000964,-0.007278,0.002227,-0.001828,0.000250,-0.008020
Slc13a4,0.027365,0.014746,0.013841,-0.003024,0.093249,-0.008859,0.038991,0.045252,-0.016012,-0.001215,...,0.017168,0.040931,-0.000292,0.004044,-0.003103,-0.006596,0.007734,-0.000833,0.000168,-0.009801
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Atp5g2,0.029159,-0.002025,-0.019998,-0.007278,-0.006596,0.004634,-0.024253,-0.062036,0.017316,-0.004337,...,-0.008919,-0.032618,0.002674,-0.010208,-0.011663,0.197001,-0.019517,-0.014945,-0.005344,0.012151
Ptdss1,-0.005043,0.002249,0.010930,0.002227,0.007734,0.010938,0.024716,0.043201,-0.006730,0.001509,...,0.007419,0.023149,-0.000251,0.006186,0.005790,-0.019517,0.102351,0.008030,0.003133,-0.004160
Dock4,-0.028147,-0.000134,-0.001086,-0.001828,-0.000833,0.004271,-0.009652,0.015988,-0.002998,0.000749,...,0.002273,0.012844,0.000694,0.000625,0.016466,-0.014945,0.008030,0.059159,0.003752,0.000763
Cpt1a,-0.008071,0.000137,0.001604,0.000250,0.000168,0.000979,-0.001799,0.007911,-0.001544,0.000242,...,0.001096,0.005033,0.000097,0.000656,0.003521,-0.005344,0.003133,0.003752,0.011372,-0.000400


In [7]:
# Simulate expression changes based on a correlation matrix
def adjust_expression_based_on_correlation(expression_df, corr_matrix, tf_name, adjustment_factor, max_depth=5):
    """
    Adjusts the expression of a specific gene and propagates its effect
    to downstream target genes using a gene co-expression network, with a limit on propagation depth.

    :param expression_df: Gene expression matrix (DataFrame), rows represent cells and columns represent genes
    :param corr_matrix: Gene-gene correlation coefficient matrix
    :param tf_name: Name of the transcription factor to be perturbed (string)
    :param adjustment_factor: Log-scale fold change to apply to the TF expression (positive to increase, negative to reduce)
    :param max_depth: Maximum propagation depth for downstream effects
    :return: Adjusted expression matrix (DataFrame)
    """
    # Step 1: Replace zero values with small random values (to avoid division by zero)
    random_offsets = np.random.uniform(0, 0.4, size=expression_df.shape)  # Generate random offsets
    expression_df = expression_df.astype(float)  # Ensure all values are float
    expression_df += np.where(expression_df == 0, random_offsets, 0)  # Replace zero values with small random numbers

    # Step 2: Adjust expression of the starting transcription factor
    initial_expression = expression_df[tf_name].copy().values.astype(float)  # Save original expression
    expression_df[tf_name] = expression_df[tf_name].values.astype(float) * 10**adjustment_factor  # Apply fold change
    updated_expression = expression_df[tf_name]  # Get updated expression

    # Calculate average relative change
    initial_change_value = (updated_expression / initial_expression) - 1  # Compute relative change
    gene_changes = {tf_name: initial_change_value.mean()}  # Initialize change dict with TF

    # Step 3: Propagate changes to correlated genes using the correlation matrix
    def propagate_gene_expression_change(corr_matrix, gene_changes, max_depth=5):
        propagated_changes = gene_changes.copy()  # Initialize propagated changes
        visited_genes = set(gene_changes.keys())  # Track visited genes to avoid cycles

        def propagate(current_changes, depth):
            if depth >= max_depth:
                return
            next_changes = {}
            # For each currently changed gene, propagate to correlated targets
            for gene, change in current_changes.items():
                if gene not in corr_matrix.index:
                    continue
                for target_gene in corr_matrix.columns:
                    if gene != target_gene and target_gene not in visited_genes:
                        corr_value = corr_matrix.loc[gene, target_gene]
                        influence = corr_value * change  # Compute influence based on correlation
                        if target_gene in propagated_changes:
                            propagated_changes[target_gene] += influence
                        else:
                            propagated_changes[target_gene] = influence
                        if target_gene in next_changes:
                            next_changes[target_gene] += influence
                        else:
                            next_changes[target_gene] = influence
            visited_genes.update(current_changes.keys())
            visited_genes.update(next_changes.keys())
            if next_changes:
                propagate(next_changes, depth + 1)

        # Start propagation
        propagate(gene_changes, depth=0)
        return propagated_changes

    # Perform propagation based on correlation structure
    propagated_changes = propagate_gene_expression_change(corr_matrix, gene_changes, max_depth=max_depth)

    # Step 4: Apply propagated changes to the expression matrix
    for gene, change in propagated_changes.items():
        if gene in expression_df.columns and gene not in gene_changes:  # Avoid modifying the TF again
            expression_df[gene] *= (1 + change)
    expression_df -= 0.4  # Offset correction
    expression_df[expression_df < 0] = 0  # Clamp negative values to zero

    return expression_df


In [8]:
###calculate result
adjusted_df = adjust_expression_based_on_correlation(expression_df, cov_matrix, 'Cebpa',6, max_depth=5)

In [12]:
# Check expression level of the gene of interest (goi) before overexpression
goi="Tfap2c"
print(f"{goi}过表达前表达量：")
print(expression_df[goi].describe())
# Check expression level of the GOI after overexpression
print(f"{goi}过表达后表达量：")
print(adjusted_df[goi].describe())

Tfap2c过表达前表达量：
count    500.000000
mean       0.001407
std        0.023491
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        0.471904
Name: Tfap2c, dtype: float64
Tfap2c过表达后表达量：
count     500.000000
mean     1143.800070
std       686.280315
min         0.617109
25%       542.995982
50%      1157.691065
75%      1748.685908
max      2771.644676
Name: Tfap2c, dtype: float64


In [10]:
adjusted_df.to_csv('Cebpa_OE.csv')